# Liu2024 S-JEPA PreLocal: SSL-Held-Out Lv14 Evaluation

Leakage-safe within-subject PreLocal decoding on the 14 patients excluded from Liu2024 SSL pretraining. The local feature encoder is frozen; only `spatial_conv` and the classifier head adapt within each outer training fold.

# 1. Setup

In [ ]:
import builtins
import hashlib
import json
import platform
import sys
from datetime import datetime
from pathlib import Path

import braindecode
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import sklearn
import torch
from scipy.stats import wilcoxon

WORKING_DIR = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(WORKING_DIR / 'src' / 'liu2024'))
import liu2024_prelocal_clean as clean

mne.set_log_level('WARNING')
print(f'Python: {sys.version.split()[0]} | platform: {platform.platform()}')
print(f'Working directory: {WORKING_DIR}')

# 2. Configuration
## 2.1 Locked Cohort

In [ ]:
LV14 = [1, 3, 7, 9, 10, 11, 14, 15, 17, 29, 31, 32, 37, 41]
SSL_TRAIN30 = [2, 4, 5, 6, 8, 13, 16, 18, 20, 21, 23, 24, 25, 26, 27, 28, 33, 34, 35, 36, 38, 39, 40, 43, 44, 45, 46, 48, 49, 50]
SSL_VAL6 = [12, 19, 22, 30, 42, 47]
assert set(LV14) | set(SSL_TRAIN30) | set(SSL_VAL6) == set(range(1, 51))
assert not (set(LV14) & set(SSL_TRAIN30) or set(LV14) & set(SSL_VAL6) or set(SSL_TRAIN30) & set(SSL_VAL6))

## 2.2 CONFIG

In [ ]:
CONFIG = {
    # Paths / run identity
    'enabled': False,
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-sjepa-prelocal-lv14-ssl-holdout'),
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'experiment_name': 'sjepa_prelocal_lv14_ssl_holdout',
    'config_note': 'Template only: supply the completed corrected SSL student_backbone_best.pt and SHA256.',

    # Dataset / preprocessing
    'subjects_to_use': LV14,
    'allow_subset_for_smoke': False,
    'target_sfreq': 128,
    'mi_window_seconds': 4.0,
    'average_reference': True,
    'bandpass_hz': [0.5, 40.0],
    'normalization_mode': 'none',
    'normalization_eps': 1e-6,

    # Required local SSL checkpoint
    'model_name': 'SignalJEPA_PreLocal',
    'pretrained_checkpoint_path': None,
    'pretrained_checkpoint_sha256': None,
    'require_validated_pretraining_export': True,
    'require_checkpoint_sha256': True,
    'require_best_checkpoint_filename': True,
    'allow_untrained_checkpoint_for_smoke': False,
    'allow_incomplete_split_for_smoke': False,
    'expected_ssl_train_subject_ids': SSL_TRAIN30,
    'expected_ssl_val_subject_ids': SSL_VAL6,
    'expected_ssl_excluded_subject_ids': LV14,
    'strategy': 'new',

    # Evaluation / training
    'validation_only': False,
    'cv_folds': 5,
    'cv_seed': 2026,
    'val_fraction': 0.2,
    'val_seed': 2026,
    'batch_size': 16,
    'n_epochs': 5000,
    'early_stopping_patience': 50,
    'early_stopping_threshold': 0.0,
    'learning_rate': 0.0005,
    'weight_decay': 0.0,

    # Reproducibility / inference
    'seed': 2026,
    'set_seed': True,
    'collapse_threshold': 0.875,
    'bootstrap_iterations': 10000,
}


In [ ]:
if not CONFIG.get('enabled', False):
    raise RuntimeError('Template disabled: provide the completed checkpoint path/SHA in an override config and set enabled=true.')
if CONFIG['strategy'] != 'new' or CONFIG['mi_window_seconds'] != 4.0 or CONFIG['target_sfreq'] != 128:
    raise ValueError('This evaluation is locked to PreLocal strategy=new and marker-relative 0-4 s at 128 Hz.')
if not CONFIG.get('validation_only', False) and CONFIG.get('allow_subset_for_smoke', False):
    raise ValueError('allow_subset_for_smoke is permitted only with validation_only=true.')
if not CONFIG.get('validation_only', False) and sorted(CONFIG['subjects_to_use']) != LV14:
    raise ValueError('The full evaluation requires exactly the canonical Lv14 cohort.')
print(json.dumps(CONFIG, indent=2))

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    digest = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f'{timestamp}_{digest}'

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG['artifact_dir']) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / 'run.log'
_LOG_FILE_HANDLE = open(LOG_PATH, 'a', buffering=1, encoding='utf-8', errors='replace')

def _safe_write(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, 'encoding', None) or 'utf-8'
        stream.write(text.encode(encoding, errors='replace').decode(encoding, errors='replace'))

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop('sep', ' '); end = kwargs.pop('end', chr(10)); flush = kwargs.pop('flush', False); target = kwargs.pop('file', None) or sys.stdout
    message = sep.join(str(arg) for arg in args)
    text = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}{end}"
    _safe_write(target, text); _safe_write(_LOG_FILE_HANDLE, text)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / 'config.json'
config_path.write_text(json.dumps(CONFIG, indent=2, allow_nan=False), encoding='utf-8')
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')
print(f'Config:     {config_path}')

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.cuda.is_available(): return torch.device('cuda')
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device('mps')
    return torch.device('cpu')
DEVICE = resolve_device()
BASE_SEED = int(CONFIG['seed'])
if CONFIG['set_seed']: clean.seed_everything(BASE_SEED)
print(f'Using device: {DEVICE} | seed: {BASE_SEED}')

# 3. Load and Prepare Data
## 3.1 Checkpoint Provenance Before Outcomes

In [ ]:
CHECKPOINT_PAYLOAD, CHECKPOINT_AUDIT = clean.validate_pretraining_export(CONFIG, 512)
checkpoint_provenance_path = ARTIFACT_DIR / 'checkpoint_provenance.json'
checkpoint_provenance_path.write_text(json.dumps(CHECKPOINT_AUDIT, indent=2, allow_nan=False), encoding='utf-8')
print(f"Validated SSL checkpoint epoch {CHECKPOINT_AUDIT['checkpoint_epoch']} SHA256={CHECKPOINT_AUDIT['checkpoint_sha256']}")

## 3.2 Exact-Marker Independent-Trial Preprocessing

In [ ]:
source_root = Path(CONFIG['source_extract_dir'])
requested = sorted(int(x) for x in CONFIG['subjects_to_use'])
all_paths = sorted(source_root.glob('sub-*/sub-*_task-motor-imagery_eeg.mat'))
paths = [path for path in all_paths if clean.subject_id_from_path(path) in set(requested)]
observed = sorted(clean.subject_id_from_path(path) for path in paths)
if observed != requested:
    raise RuntimeError(f'Requested/loaded cohort mismatch: requested={requested}, loaded={observed}')
if not CONFIG.get('allow_subset_for_smoke', False) and observed != LV14:
    raise RuntimeError(f'Full run requires exact Lv14, loaded={observed}')

SUBJECT_DATA = {}
inventory_rows, marker_rows = [], []
for path in paths:
    subject = clean.load_subject(path)
    sid = str(subject['subject_id'])
    x, records = clean.preprocess_subject(subject, CONFIG)
    if x.shape != (40, 29, 512): raise RuntimeError(f'{sid}: unexpected preprocessed shape {x.shape}')
    SUBJECT_DATA[sid] = {'x': x, 'y': subject['labels'].copy()}
    inventory_rows.append({'subject_id': sid, 'source_path': subject['path'], 'source_sha256': subject['source_sha256'], 'preprocessed_shape': list(x.shape), 'class_counts': np.bincount(subject['labels'], minlength=2).tolist()})
    marker_rows.extend(records)
SUBJECTS = sorted(SUBJECT_DATA, key=int)
subject_inventory_path = ARTIFACT_DIR / 'subject_inventory.csv'
marker_inventory_path = ARTIFACT_DIR / 'trial_marker_inventory.csv'
pd.DataFrame(inventory_rows).to_csv(subject_inventory_path, index=False)
pd.DataFrame(marker_rows).to_csv(marker_inventory_path, index=False)
print(f'Loaded subjects: {SUBJECTS} | trials={len(marker_rows)}')

## 3.3 Dataset Classes
The shared helper creates fold-local `TensorDataset` instances only after outer and inner indices are fixed.

## 3.4 Data Inventory
Source hashes, exact markers, class counts, and preprocessed shapes are persisted before fitting.

# 4. Model
## 4.1 Frozen Local Encoder, Trainable Adapter and Head

In [ ]:
probe_model, MODEL_AUDIT = clean.build_model(CONFIG, 512)
if MODEL_AUDIT['checkpoint_sha256'] != CHECKPOINT_AUDIT['checkpoint_sha256']:
    raise RuntimeError('Probe model checkpoint provenance drifted after preflight')
model_audit_path = ARTIFACT_DIR / 'model_load_audit.json'
model_audit_path.write_text(json.dumps(MODEL_AUDIT, indent=2, allow_nan=False), encoding='utf-8')
print(f"Loaded {len(MODEL_AUDIT['loaded_keys'])} local encoder keys; ignored {len(MODEL_AUDIT['ignored_checkpoint_keys'])} contextual keys.")
print(f"Trainable parameters: {MODEL_AUDIT['trainable']}")
del probe_model

## 4.2 Diagnostics
Every fold records loaded checkpoint provenance, trainable names, tested-state hash, selected epoch, and collapse diagnostics.

# 5. Training
## 5.1 Explicit Inner Validation
## 5.2 Within-Subject Five-Fold Runner

In [ ]:
FOLD_RESULTS = []
if CONFIG.get('validation_only', False):
    print('Validation-only mode: checkpoint/model/data checks passed; no downstream fitting performed.')
else:
    for sid in SUBJECTS:
        data = SUBJECT_DATA[sid]
        for split in clean.make_outer_splits(data['y'], CONFIG):
            print(f'Subject {sid} fold {split["fold_id"]}/{CONFIG["cv_folds"]}')
            result = clean.run_fold(int(sid), data['x'], data['y'], split, CONFIG, DEVICE)
            audit = result['model_audit']
            result['model_audit'] = {'checkpoint_sha256': audit['checkpoint_sha256'], 'checkpoint_epoch': audit['checkpoint_epoch'], 'loaded_keys': audit['loaded_keys'], 'trainable': audit['trainable'], 'n_trainable': audit['n_trainable']}
            FOLD_RESULTS.append(result)
            print(f"  BA={result['balanced_accuracy']:.3f} best_epoch={result['best_epoch']} pred={result['prediction_histogram']}")

## 5.3 Completion Assertions

In [ ]:
if not CONFIG.get('validation_only', False):
    if SUBJECTS != [str(x) for x in LV14]: raise RuntimeError(f'Unexpected final cohort: {SUBJECTS}')
    if len(FOLD_RESULTS) != 70: raise RuntimeError(f'Expected 70 folds, got {len(FOLD_RESULTS)}')
    if any(row['model_audit']['checkpoint_sha256'] != CHECKPOINT_AUDIT['checkpoint_sha256'] for row in FOLD_RESULTS): raise RuntimeError('Fold checkpoint SHA drift')
    if any(row['outer_test_used_for_fit'] or row['outer_test_used_for_selection'] for row in FOLD_RESULTS): raise RuntimeError('Outer-test boundary violation')
    print('Completion assertions: 14 subjects, 70 folds, fixed checkpoint, no outer-test fitting/selection.')

# 6. Results
## 6.1 Exact-Once OOF Aggregation

In [ ]:
SUBJECT_METRICS, GLOBAL_METRICS, TRIAL_PREDICTIONS = {}, {}, []
if FOLD_RESULTS:
    labels_by_subject = {sid: SUBJECT_DATA[sid]['y'] for sid in SUBJECTS}
    SUBJECT_METRICS, GLOBAL_METRICS, TRIAL_PREDICTIONS = clean.aggregate(FOLD_RESULTS, labels_by_subject)
    if GLOBAL_METRICS['n_original_trial_predictions'] != 560: raise RuntimeError('Expected exactly 560 OOF predictions')
    values = np.asarray([SUBJECT_METRICS[str(s)]['balanced_accuracy'] for s in LV14])
    rng = np.random.default_rng(BASE_SEED)
    bootstrap = rng.choice(values, size=(int(CONFIG['bootstrap_iterations']), len(values)), replace=True).mean(axis=1)
    GLOBAL_METRICS['subject_bootstrap_95_ci'] = [float(x) for x in np.quantile(bootstrap, [0.025, 0.975])]
    try:
        GLOBAL_METRICS['wilcoxon_vs_chance_p'] = float(wilcoxon(values - 0.5, alternative='two-sided').pvalue)
    except ValueError:
        GLOBAL_METRICS['wilcoxon_vs_chance_p'] = None
    GLOBAL_METRICS['single_class_folds'] = int(sum(row['collapse_diagnostics']['single_class_prediction'] for row in FOLD_RESULTS))
    GLOBAL_METRICS['near_collapse_folds'] = int(sum(row['collapse_diagnostics']['near_collapse_7_of_8'] for row in FOLD_RESULTS))
    print(json.dumps(GLOBAL_METRICS, indent=2, allow_nan=False))

## 6.2 Performance Visualizations

In [ ]:
plot_paths = {}
if FOLD_RESULTS:
    subject_plot = ARTIFACT_DIR / 'lv14_prelocal_subject_performance.png'
    values = [100 * SUBJECT_METRICS[str(s)]['balanced_accuracy'] for s in LV14]
    fig, ax = plt.subplots(figsize=(10, 4)); ax.bar([str(s) for s in LV14], values, color='#355070'); ax.axhline(50, color='black', ls='--', lw=1); ax.set(xlabel='Liu subject', ylabel='Exactly-once OOF BA (%)', ylim=(0, 100)); fig.tight_layout(); fig.savefig(subject_plot, dpi=160); plt.close(fig)
    global_plot = ARTIFACT_DIR / 'lv14_prelocal_global_performance.png'
    mean = 100 * GLOBAL_METRICS['mean_subject_balanced_accuracy']; ci = 100 * np.asarray(GLOBAL_METRICS['subject_bootstrap_95_ci'])
    fig, ax = plt.subplots(figsize=(4, 4)); ax.bar(['PreLocal'], [mean], color='#355070', yerr=[[mean-ci[0]], [ci[1]-mean]], capsize=5); ax.axhline(50, color='black', ls='--'); ax.set(ylabel='Mean subject BA (%)', ylim=(0, 100)); fig.tight_layout(); fig.savefig(global_plot, dpi=160); plt.close(fig)
    confusion_plot = ARTIFACT_DIR / 'lv14_prelocal_confusion_matrix.png'
    cm = sum((np.asarray(row['confusion_matrix']) for row in FOLD_RESULTS), np.zeros((2, 2), dtype=int))
    fig, ax = plt.subplots(figsize=(4, 4)); ax.imshow(cm, cmap='Blues'); [ax.text(j, i, str(cm[i, j]), ha='center', va='center') for i in range(2) for j in range(2)]; ax.set(xlabel='Predicted', ylabel='True', xticks=[0, 1], yticks=[0, 1]); fig.tight_layout(); fig.savefig(confusion_plot, dpi=160); plt.close(fig)
    plot_paths = {'subject_plot': str(subject_plot), 'global_plot': str(global_plot), 'confusion_plot': str(confusion_plot)}

## 6.3 Experiment Summary
## 6.4 Model-Specific Analysis
The run-level model audit explicitly lists the seven loaded local feature-encoder keys and all ignored contextual-backbone keys.

## 6.5 Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / 'cv_results.json'
subject_metrics_path = ARTIFACT_DIR / 'subject_metrics.json'
global_metrics_path = ARTIFACT_DIR / 'global_metrics.json'
trial_predictions_path = ARTIFACT_DIR / 'trial_predictions.csv'
split_indices_path = ARTIFACT_DIR / 'split_indices.json'
cv_results_path.write_text(json.dumps(FOLD_RESULTS, indent=2, allow_nan=False), encoding='utf-8')
subject_metrics_path.write_text(json.dumps(SUBJECT_METRICS, indent=2, allow_nan=False), encoding='utf-8')
global_metrics_path.write_text(json.dumps(GLOBAL_METRICS, indent=2, allow_nan=False), encoding='utf-8')
pd.DataFrame(TRIAL_PREDICTIONS).to_csv(trial_predictions_path, index=False)
split_payload = [{key: row[key] for key in ['subject_id', 'fold_id', 'train_indices', 'inner_train_indices', 'validation_indices', 'test_indices']} for row in FOLD_RESULTS]
split_indices_path.write_text(json.dumps(split_payload, indent=2, allow_nan=False), encoding='utf-8')
module_path = Path(clean.__file__).resolve()
notebook_path = WORKING_DIR / 'src' / 'liu2024' / 'liu2024_sjepa_prelocal_lv14_ssl_holdout.ipynb'
completion_path = ARTIFACT_DIR / 'COMPLETED.json'
artifacts = {'config': str(config_path), 'run_log': str(LOG_PATH), 'checkpoint_provenance': str(checkpoint_provenance_path), 'model_load_audit': str(model_audit_path), 'subject_inventory': str(subject_inventory_path), 'marker_inventory': str(marker_inventory_path), 'cv_results': str(cv_results_path), 'subject_metrics': str(subject_metrics_path), 'global_metrics': str(global_metrics_path), 'trial_predictions': str(trial_predictions_path), 'split_indices': str(split_indices_path), 'completion_manifest': str(completion_path), **plot_paths}
run_metadata = {
    'run_id': RUN_ID, 'artifact_dir': str(ARTIFACT_DIR), 'experiment_name': CONFIG['experiment_name'], 'config_note': CONFIG['config_note'], 'validation_only': bool(CONFIG['validation_only']),
    'subjects': SUBJECTS, 'expected_lv14': [str(x) for x in LV14], 'n_channels': 29, 'channel_names': clean.LIU_EEG_NAMES, 'model_name': CONFIG['model_name'], 'strategy': CONFIG['strategy'],
    'checkpoint_provenance': CHECKPOINT_AUDIT, 'model_load_audit': MODEL_AUDIT,
    'preprocessing_contract': 'independent complete-trial average reference/FIR 0.5-40 Hz/resample, exact marker-2 crop [0,4)s, microvolts',
    'implementation_module': str(module_path), 'implementation_sha256': clean.sha256_file(module_path), 'notebook_sha256': clean.sha256_file(notebook_path),
    'versions': {'python': sys.version, 'torch': torch.__version__, 'braindecode': braindecode.__version__, 'mne': mne.__version__, 'sklearn': sklearn.__version__},
    'seed': BASE_SEED, 'cv_seed': CONFIG['cv_seed'], 'val_seed': CONFIG['val_seed'], 'global_metrics': GLOBAL_METRICS,
    'estimand': 'within-subject calibrated decoding on patients excluded from SSL pretraining; not zero-shot patient transfer',
    'leakage_assertions': {'lv14_excluded_from_ssl_train_and_validation': True, 'independent_trial_preprocessing': True, 'exact_marker_required': True, 'normalizer_fit_inner_train_only': True, 'outer_test_used_for_fit': False, 'outer_test_used_for_selection': False, 'exact_once_oof': bool(FOLD_RESULTS)},
    'artifacts': artifacts,
}
run_metadata_path = ARTIFACT_DIR / 'run_metadata.json'
run_metadata['artifacts']['run_metadata'] = str(run_metadata_path)
run_metadata_path.write_text(json.dumps(run_metadata, indent=2, allow_nan=False), encoding='utf-8')
completion = {'run_id': RUN_ID, 'validation_only': bool(CONFIG['validation_only']), 'n_subjects': len(SUBJECTS), 'n_folds': len(FOLD_RESULTS), 'n_predictions': len(TRIAL_PREDICTIONS), 'checkpoint_sha256': CHECKPOINT_AUDIT['checkpoint_sha256']}
completion_path.write_text(json.dumps(completion, indent=2, allow_nan=False), encoding='utf-8')
print(f'Run metadata saved to: {run_metadata_path}')
print(f'Completion manifest:   {completion_path}')
print(f'\nAll artifacts in: {ARTIFACT_DIR}')
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass